# Stata Analysis Code Template

This is a template for Stata analysis code. It is designed to be used in a Jupyter notebook.

* Key rules: 
    - EVERY cell should be self-contained and executable, so all global settings should be repeated in every cell.
    - Each cell usually write one table, panel or figure.
    - Do not write custom `program define` wrappers, Mata code, or meta/diagnostic machinery (postfiles, model-metadata logging, coefficient dumps); call estimators directly and store results with `est store` (plus `estadd` for table rows).
    - Keep the codes as few lines as possible, minimize unnecessary code beyond shown in the examples below.

* Output rules (always follow):
    - Tables: export to four formats with the same basename — `.md`, `.rtf`, `.csv`, `.tex` — using the `esttab` options shown below (`cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label`; add `r2_p` for nonlinear models).
    - Figures: export to both `.pdf` and `.png`.
    - No extra side files (`.ster`, metadata, diagnostic dumps); the four formats are the deliverable.

* Entry file: one `reg.dta` file per sample (one for a project that uses only one baseline sample)

## Example: baseline models

In [ ]:
set more off

global y "price"
global x "mpg"
global ctrl "weight foreign"
global fe "rep78"
global xv "headroom trunk"
global cl "make"


sysuse "auto.dta", clear // replace with `use "reg.dta", clear` in a real project

est clear
local cnt = 1
foreach dep_var of varlist $y {
    foreach indep_var of varlist $x {
        reghdfe `dep_var' `indep_var' $ctrl, absorb($fe) vce(cluster $cl)
        est store m`cnt'
        local ++cnt
    }
}

// Four formats each set of results
esttab m* using "results.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label // add r2_p if nonlinear regression
esttab m* using "results.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label // add r2_p if nonlinear regression
esttab m* using "results.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label // add r2_p if nonlinear regression
esttab m* using "results.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label // add r2_p if nonlinear regression
est clear









(1978 automobile data)



(MWFE estimator converged in 1 iterations)

HDFE Linear regression                            Number of obs   =         69
Absorbing 1 HDFE group                            F(   3,     61) =      14.03
Statistics robust to heteroskedasticity           Prob > F        =     0.0000
                                                  R-squared       =     0.5018
                                                  Adj R-squared   =     0.4446
                                                  Within R-sq.    =     0.4944
Number of clusters (make)    =         69         Root MSE        =  2170.5469

                                  (Std. err. adjusted for 69 clusters in make)
------------------------------------------------------------------------------
             |               Robust
       price | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
         mpg |   

## Example: cross-sectional tests

In [ ]:
set more off

global y "price"
global x "mpg"
global ctrl "weight foreign"
global fe "rep78"
global xv "headroom trunk"
global cl "make"


sysuse "auto.dta", clear // replace with `use "reg.dta", clear` in a real project

est clear
local cnt = 1
foreach dep_var of varlist $y {
    foreach indep_var of varlist $x {
        foreach cross_var of varlist $xv {
            reghdfe `dep_var' c.`indep_var'##c.`cross_var' $ctrl, absorb($fe) vce(cluster $cl)
            est store m`cnt'
            local ++cnt
        }
    }
}

// Four formats each set of results
esttab m* using "results.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label // add r2_p if nonlinear regression
esttab m* using "results.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label // add r2_p if nonlinear regression
esttab m* using "results.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label // add r2_p if nonlinear regression
esttab m* using "results.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label // add r2_p if nonlinear regression
est clear









(1978 automobile data)



(MWFE estimator converged in 1 iterations)

HDFE Linear regression                            Number of obs   =         69
Absorbing 1 HDFE group                            F(   5,     59) =       9.09
Statistics robust to heteroskedasticity           Prob > F        =     0.0000
                                                  R-squared       =     0.5356
                                                  Adj R-squared   =     0.4647
                                                  Within R-sq.    =     0.5287
Number of clusters (make)    =         69         Root MSE        =  2130.8113

                                  (Std. err. adjusted for 69 clusters in make)
------------------------------------------------------------------------------
             |               Robust
       price | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
         mpg |   

## Example: descriptive statistics

In [ ]:
set more off

global y "price"
global x "mpg"
global ctrl "weight foreign"
global fe "rep78"
global xv "headroom trunk"
global cl "make"


sysuse "auto.dta", clear // replace with `use "reg.dta", clear` in a real project

est clear

estpost tabstat $y $x $ctrl, stat(N mean sd p25 p50 p75) c(s)
esttab using "descriptive.csv", replace ///
    cells("count(fmt(%9.0f)) mean(fmt(%9.3f)) sd(fmt(%9.3f)) p25(fmt(%9.3f)) p50(fmt(%9.3f)) p75(fmt(%9.3f))") label
est clear










(1978 automobile data)



Summary statistics: count mean sd p25 p50 p75
     for variables: price mpg weight foreign

             |  e(count)    e(mean)      e(sd)     e(p25)     e(p50)     e(p75) 
-------------+------------------------------------------------------------------
       price |        74   6165.257   2949.496       4195     5006.5       6342 
         mpg |        74    21.2973   5.785503         18         20         25 
      weight |        74   3019.459   777.1936       2240       3190       3600 
     foreign |        74   .2972973   .4601885          0          0          1 

(output written to descriptive.csv)

